# 02. Time Cuts And Memory

This notebook shows one of AETHER's core ideas: the system can answer both **what is true now** and **what was true at an earlier journal cut**.

We will append a few status changes, inspect the journal, and query the same predicate at multiple cuts.


In [ ]:
from pathlib import Path
import subprocess
import sys

candidate_roots = [Path.cwd().resolve(), *Path.cwd().resolve().parents]
repo_root = next(
    (
        path
        for path in candidate_roots
        if (path / "python").exists() and (path / "Cargo.toml").exists()
    ),
    None,
)

if repo_root is None:
    repo_root = Path("/content/AETHER")
    if not repo_root.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", "https://github.com/fyremael/AETHER", str(repo_root)],
            check=True,
        )

python_root = repo_root / "python"
if str(python_root) not in sys.path:
    sys.path.insert(0, str(python_root))

repo_root


In [ ]:
from notebooks.colab_setup import ensure_rust_toolchain, pretty_json, start_http_service, stop_http_service
from aether_sdk import AetherClient, make_datom, value_string

ensure_rust_toolchain()
session = start_http_service(repo_root)
client = AetherClient(session.base_url)

pretty_json(client.health())


## Append A Tiny Status History

The element IDs `e1`, `e2`, `e3`, and `e4` become stable cut points we can replay later.


In [ ]:
client.append(
    [
        make_datom(entity=1, attribute=1, value=value_string("queued"), element=1),
        make_datom(entity=1, attribute=1, value=value_string("running"), element=2),
        make_datom(entity=2, attribute=1, value=value_string("queued"), element=3),
        make_datom(entity=2, attribute=1, value=value_string("ready"), element=4),
    ]
)

history = client.history()
pretty_json(history)


In [ ]:
timeline_document = """
schema v1 {
  attr task.status: ScalarLWW<String>
}

predicates {
  task_status(Entity, String)
}

query status_at_e1 {
  as_of e1
  goal task_status(task, status)
  keep task, status
}

query status_at_e2 {
  as_of e2
  goal task_status(task, status)
  keep task, status
}

query status_now {
  current
  goal task_status(task, status)
  keep task, status
}
"""

timeline = client.run_document(timeline_document)
pretty_json(timeline["queries"])


In [ ]:
for query_name in ["status_at_e1", "status_at_e2", "status_now"]:
    rows = client.run_named_query(timeline_document, query_name=query_name)["rows"]
    print(query_name)
    print([row["values"] for row in rows])
    print()


## What Changed Across Cuts?

- At `AsOf(e1)`, only the first status exists.
- At `AsOf(e2)`, task `1` has already moved to `running`.
- At `Current`, both tasks reflect their latest committed values.

That is the key AETHER promise: replay is part of the semantic model, not a debugging trick.


In [ ]:
# Uncomment this cell when you are done with the notebook.
# stop_http_service(session)
